# Load data for GNN PROTAC splitter - Node classifier - Experimental Architectures

# ToDo

1. Make test set from AZ data. Make AZ data which is in public and AZ data which is not in public.

1.5 Make test set based on "cluster-leave-one-out"

1.75 5-fold cross validation - Check if it is overfitting or not

1.8 Other methods to see if the GNN overfits or not

2. Fix points of improvement for BoundaryNode GNN. Notes down below.

2.5 Make a dataframe for all evaluation metrics. Export to CSV and analyze it in Excel.
    * Is it correct to say that some protacs have "no" linker? Some have their length of their linker equal to zero.
        IF it is incorrect, then the dataprocessing may need revisions.

3. Clean up code.



1. Generate a complete dataset of PROTACs of all combinations between POI, Linker, E3
    1.1 Use a smaller subset of these rather than generating new ones.

2. Save the set of smiles for the anonymous MS for each substructure. Use these when generating the test set.

3. Make the nameing from Stefanos and Evas code congruent. Standardize mine and theirs.

4. Have the PROTACdataset class call on the same function as prepare_data. So If I update the function, it automatically applies to both. To prevent mismatch / outdated functions.

5. Retry adding rings to linker an applying MS

6. Make the subset selection of the complete set of random PROTACs to be random but reproducible with a seed.



In [1]:
# Standard library imports
import os
import sys
import io

# Data handling and scientific computing
import pandas as pd
import numpy as np
import scipy.sparse as sp
from scipy.linalg import inv

# PyTorch and related libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import random_split
from torch_geometric.nn import GCNConv, NNConv
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader

# RDKit for cheminformatics
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, GetPeriodicTable
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdchem
from rdkit.Chem import rdmolops
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem import rdMolHash


# NetworkX for network analysis
import networkx as nx

# Matplotlib for plotting and visualization
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap

# Other utilities
from tqdm import tqdm
from PIL import Image

# External datasets
from datasets import load_dataset


import re
import random


from IPython.display import display

sys.path.append('./Code/TestingGround/src/models/')

/home/knkn308/.conda/envs/env-protac-toolkit/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# User inputs:
Note that new datasets (as in a new "dataname") have all descriptors calculated from scratch, which is then saved. If there exists a dataset with a given dataname, it is retrived instead.

In [2]:
Highlighted_graph_descriptors = ['betweenness', 'local_betweenness_5', 'local_eigenvectors_x', 'local_normAvgShortestPath_5', 'local_prod_betweenness_avgShortestPath_4']   ##Which graph descriptors to calculate for the data and use to train the GNN model. There exists more graph descriptors
Existing_datanames = []



#user input
graph_descriptor_list = []#Highlighted_graph_descriptors
dataname = "NoGraphDesc_20K"  #Existing_datanames[0] # Which dataset to use
num_random_protacs = 20000
name_dataset_aibe = "80-20-split"


# Core functions 

## General functions

In [3]:
from all_functions import *

## Graph descriptor functions

More graph descriptors:
https://en.wikipedia.org/wiki/Centrality: Percolation centrality, Harmonic centrality, Freeman centralization, Dissimilarity-based centrality measures



Established node descriptors: bonacich_custom, betweenness_custom, eigenvector_custom, katz_custom      
Personal node descriptors: ovality, eccentricity, major_minor_axis_ratio, general_wiener_vector

Global -> Node pair: general_wiener_matrix, general_modified_wiener_matrix 

Global descriptors: wiener_index, hyper_wiener_index, general_modified_wiener_index, average_shortest_path_custom

### Dataset processing functions - WIP Multiple substructure matches

In [4]:
#Further criteria for legal nodes: The the POI-L node and Linker nodes can't be a part of the same ring => Calculate the resulting nodes for the linker, POI and E3. ...
# ... See if any ring has nodes from two classes. If a ring has nodes from Linker and E3, then the E3-L choice was poor. If the ring has nodes from Linker and POI, the POI-L choice was poor.
#def will_boundary_nodes_split_rings(mol, poi_L_idx, e3_l_idx):
#    ring_info = mol.GetRingInfo()
#    ring_idx = []
#    for atom in mol.GetAtoms():
#        if ring_info.IsAtomInRing(atom.GetIdx()) and (atom.GetIdx() != poi_L_idx) and (atom.GetIdx() != e3_l_idx):
#            ring_idx.append(atom.GetIdx())

# PROTACDataset - WIP The dataset class needs to be able to import CSVs


OBS Work in progress

## Download data

In [5]:
#name_dataset_aibe = "80-20-split"
dataset_aibe = load_dataset("ailab-bio/PROTAC-Substructures", name_dataset_aibe) #download
train_dataset = dataset_aibe['train']
validation_dataset = dataset_aibe['validation']

train_texts = train_dataset['text']
train_labels = train_dataset['labels']

validation_texts = validation_dataset['text']
validation_labels = validation_dataset['labels']


## Load data & remove ambigous substructure matches (matching can be improved)

In [6]:
substructures_list = train_labels + validation_labels
protac_smi_list = train_texts + validation_texts
poi_smi_r_list = []
linker_smi_r_list = []
e3_smi_r_list = []
for substructures in substructures_list:
    poi_smi_r, linker_smi_r, e3_smi_r = substructure_split_sort(substructures)
    poi_smi_r_list.append(poi_smi_r)
    linker_smi_r_list.append(linker_smi_r)
    e3_smi_r_list.append(e3_smi_r)

pub_protac_and_substructure_smi_df = pd.DataFrame({
    'smiles': protac_smi_list,
    'POI_R': poi_smi_r_list,
    'Linker_R': linker_smi_r_list,
    'E3_R': e3_smi_r_list
})

In [7]:
pub_protac_smiles_df_prepared, pub_substructure_smiles_df_prepared = prepare_data_set(pub_protac_and_substructure_smi_df, 'smiles', 'POI_R', 'Linker_R', 'E3_R')
#pub_dataset = PROTACDataset(data=pub_protac_smiles_df_prepared, substructures=pub_substructure_smiles_df_prepared, name='pub_PROTACs')

## Generate random protacs for training and validation

In [8]:
# Initialize lists to hold the split substructure components
pois = []
linkers = []
e3s = []

chosen_substructures = pub_substructure_smiles_df_prepared

# Iterate through the 'substructures' column and apply the split function
for substructure_smiles in chosen_substructures['substructures']:
    poi, linker, e3 = substructure_split_sort(substructure_smiles)
    pois.append(poi)
    linkers.append(linker)
    e3s.append(e3)

    # Create a new DataFrame with the split components
Substructuressmiles_split_df = pd.DataFrame({
    'POI': pois,
    'Linker': linkers,
    'E3': e3s
})
Substructuressmiles_split_df

,POI,Linker,E3
0,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(N...,[*:2]C(=O)CCCCCCCCCC[*:1],[*:2]NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)...
1,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(N...,[*:2]C(=O)CCCCCC[*:1],[*:2]NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)...
2,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(N...,[*:2]C(=O)CCCC[*:1],[*:2]NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)...
3,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(N...,[*:2]C(=O)CC[*:1],[*:2]NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)...
4,[*:1]c1cnc(OCC2CCC(=O)N2)c2cc(OC)c(C(N)=O)cc12,[*:2]C(=O)CCOCCOCCOCC#C[*:1],[*:2]NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)...
...,...,...,...
1795,[*:1]NC(=O)C(C#N)=Cc1ccc(OCc2ccc(C(F)(F)F)cc2C...,[*:2]CCCOCCCCOCCC[*:1],[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O
1796,[*:1]NC(=O)C(C#N)=Cc1ccc(OCc2ccc(C(F)(F)F)cc2C...,[*:2]CCOCCOCCOCCOCC[*:1],[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O
1797,[*:1]NC(=O)C(C#N)=Cc1ccc(OCc2ccc(C(F)(F)F)cc2C...,[*:2]CCOCCOCCOCCOCCOCC[*:1],[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O
1798,[*:1]c1nc(N2CCN(C(=O)C=C)CC2)c2cc(Cl)c(-c3cc(O...,[*:2]C(=O)CCOCCOCCOCCn1cc(CCCN[*:1])nn1,[*:2]Nc1cccc2c1CN(C1CCC(=O)NC1=O)C2=O


In [9]:
"""from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import ReplaceCore


def attach_rings_to_linker(mol):

    # Convert the SMILES string to an RDKit molecule
    #mol = Chem.MolFromSmiles(linker_smiles)

    # Define the ring to attach (example: benzene ring with a dummy atom)
    ring_smiles_1 = '[C:1][C:1]1=[C:1][C:1]=[C:1][C:1]=[C:1]1'#'[U:1]1=[U:1][U:1]=[U:1][U:1]=[U:1]1'  # Benzene with a dummy atom at one position
    ring_smiles_2 = '[C:2][C:2]1=[C:2][C:2]=[C:2][C:2]=[C:2]1'#'[Au:2]1=[Au:2][Au:2]=[Au:2][Au:2]=[Au:2]1'
    ring_mol_1 = Chem.MolFromSmiles(ring_smiles_1)
    ring_mol_2 = Chem.MolFromSmiles(ring_smiles_2)

    # Iterate over the atoms and find dummy atoms
    for atom in mol.GetAtoms():
        if atom.GetAtomMapNum() == 1:
            mol = AllChem.ReplaceSubstructs(mol, Chem.MolFromSmiles("[*:1]"), ring_mol_1, replacementConnectionPoint=0)[0]
        elif atom.GetAtomMapNum() == 2:
            mol = AllChem.ReplaceSubstructs(mol, Chem.MolFromSmiles("[*:2]"), ring_mol_2, replacementConnectionPoint=0)[0]


    # Convert the modified molecule back to a SMILES string
    #modified_smiles = Chem.MolToSmiles(mol)
    Chem.GetSymmSSSR(mol)  # Finding rings and re-perceiving aromaticity


    return mol



def remove_rings_from_linker(mol):
    
    "Identifies specific rings (benzene rings with a dummy atom) in the molecule and replaces them with single dummy atoms."

    "Args: linker_smiles (str): SMILES string of the linker with attached rings."

    "Returns: str: SMILES string of the linker with rings replaced by dummy atoms."

    # Convert the SMILES string to an RDKit molecule
    #mol = Chem.MolFromSmiles(linker_smiles)

    # Define the substructures to be replaced (benzene rings with a dummy atom)
    ring_substructure_1 = Chem.MolFromSmiles('[C:1][C:1]1=[C:1][C:1]=[C:1][C:1]=[C:1]1')
    ring_substructure_2 = Chem.MolFromSmiles('[C:2][C:2]1=[C:2][C:2]=[C:2][C:2]=[C:2]1')

    # Define the dummy atoms to replace the rings
    dummy_1 = Chem.MolFromSmiles('[*:1]')
    dummy_2 = Chem.MolFromSmiles('[*:2]')

    # Replace the rings with dummy atoms

    #mol = Chem.ReplaceCore(mol, ring_substructure_1, labelByIndex=True)
    mol = AllChem.ReplaceSubstructs(mol, ring_substructure_1, dummy_1, replacementConnectionPoint=0)[0]
    mol = AllChem.ReplaceSubstructs(mol, ring_substructure_2, dummy_2, replacementConnectionPoint=0)[0]

    #mol = AllChem.ReplaceSubstructs(mol, ring_substructure_2, dummy_2, replacementConnectionPoint=0)[0]
    #display(mol)


    # Convert the modified molecule back to a SMILES string
    #modified_smiles = Chem.MolToSmiles(mol)
    Chem.GetSymmSSSR(mol)  # Finding rings and re-perceiving aromaticity
    Chem.SanitizeMol(mol)

    return mol"""



# Example usage
#linker_with_dummies = "[*:1]C#CCOCC(C)OCC(c1cccc(c1C)C)OCC(C)C(=O)[*:2]"# "[*:1]C#CCOCCOCC(C1=CC=CC=C1)OCCC(=O)[*:2]"# "[*:2]C(=O)CCOCCOCCOCC#C[*:1]"  # Example linker with dummy atoms
#mol = Chem.MolFromSmiles(linker_with_dummies)
#display(mol)
#modified_linker = attach_rings_to_linker(mol)
#display(modified_linker)
#modified_linker = Chem.Scaffolds.MurckoScaffold.GetScaffoldForMol(modified_linker)
#modified_linker_inversed = remove_rings_from_linker(modified_linker)
#display(modified_linker_inversed)
#display(modified_linker_inversed)




'from rdkit import Chem\nfrom rdkit.Chem import AllChem\nfrom rdkit.Chem import ReplaceCore\n\n\ndef attach_rings_to_linker(mol):\n\n    # Convert the SMILES string to an RDKit molecule\n    #mol = Chem.MolFromSmiles(linker_smiles)\n\n    # Define the ring to attach (example: benzene ring with a dummy atom)\n    ring_smiles_1 = \'[C:1][C:1]1=[C:1][C:1]=[C:1][C:1]=[C:1]1\'#\'[U:1]1=[U:1][U:1]=[U:1][U:1]=[U:1]1\'  # Benzene with a dummy atom at one position\n    ring_smiles_2 = \'[C:2][C:2]1=[C:2][C:2]=[C:2][C:2]=[C:2]1\'#\'[Au:2]1=[Au:2][Au:2]=[Au:2][Au:2]=[Au:2]1\'\n    ring_mol_1 = Chem.MolFromSmiles(ring_smiles_1)\n    ring_mol_2 = Chem.MolFromSmiles(ring_smiles_2)\n\n    # Iterate over the atoms and find dummy atoms\n    for atom in mol.GetAtoms():\n        if atom.GetAtomMapNum() == 1:\n            mol = AllChem.ReplaceSubstructs(mol, Chem.MolFromSmiles("[*:1]"), ring_mol_1, replacementConnectionPoint=0)[0]\n        elif atom.GetAtomMapNum() == 2:\n            mol = AllChem.Replace

In [10]:
#smi = "[*:1]C#CCOCC(C)OCC(c1cccc(c1C)C)OCC(C)C(=O)[*:2]"
#display(Chem.MolFromSmiles(smi))
#a = get_anonymous_murcko(smi)
#display(Chem.MolFromSmiles(a))

In [11]:
for col in Substructuressmiles_split_df.columns:
    Substructuressmiles_split_df = generate_anonymous_murcko_scaffold(Substructuressmiles_split_df, col)

In [12]:
Substructuressmiles_split_df['POI_AnonMS'] = Substructuressmiles_split_df['POI_AnonMS'].apply(standardize_smiles)
Substructuressmiles_split_df['Linker_AnonMS'] = Substructuressmiles_split_df['Linker_AnonMS'].apply(standardize_smiles)
Substructuressmiles_split_df['E3_AnonMS'] = Substructuressmiles_split_df['E3_AnonMS'].apply(standardize_smiles)

# Extract unique standardized SMILES
unique_poi_smiles = Substructuressmiles_split_df['POI_AnonMS'].dropna().unique().tolist()
unique_linker_smiles = Substructuressmiles_split_df['Linker_AnonMS'].dropna().unique().tolist()
unique_e3_smiles = Substructuressmiles_split_df['E3_AnonMS'].dropna().unique().tolist()

print(f'len(unique_poi_smiles): {len(unique_poi_smiles)}')
print(f'len(unique_linker_smiles): {len(unique_linker_smiles)}')
print(f'len(unique_e3_smiles): {len(unique_e3_smiles)}')

len(unique_poi_smiles): 184
len(unique_linker_smiles): 605
len(unique_e3_smiles): 45


In [13]:
# Loop through each unique E3 SMILES string and visualize the molecule

#for smi in unique_poi_smiles:
#    mol = Chem.MolFromSmiles(smi)
#    display(mol)
#    i += 1
#    if i % 40 == 0:
#        break

In [14]:
# Create mappings for POI, Linker, and E3
poi_group_mapping = create_group_index_mapping(unique_poi_smiles)
linker_group_mapping = create_group_index_mapping(unique_linker_smiles)
e3_group_mapping = create_group_index_mapping(unique_e3_smiles)

# Assign group indices
Substructuressmiles_split_df['POI_Group'] = Substructuressmiles_split_df['POI_AnonMS'].map(poi_group_mapping)
Substructuressmiles_split_df['Linker_Group'] = Substructuressmiles_split_df['Linker_AnonMS'].map(linker_group_mapping)
Substructuressmiles_split_df['E3_Group'] = Substructuressmiles_split_df['E3_AnonMS'].map(e3_group_mapping)



# Collect unique substructures for POIs, linkers, and E3s
unique_poi_substructures = collect_unique_substructures(Substructuressmiles_split_df, 'POI_Group', 'POI')
unique_linker_substructures = collect_unique_substructures(Substructuressmiles_split_df, 'Linker_Group', 'Linker')
unique_e3_substructures = collect_unique_substructures(Substructuressmiles_split_df, 'E3_Group', 'E3')



In [15]:

# Select random substructures
random_poi = select_random_substructures(unique_poi_substructures)
random_linker = select_random_substructures(unique_linker_substructures)
random_e3 = select_random_substructures(unique_e3_substructures)

# Print the selected random substructures
print("Random POI Substructure:", random_poi)
print("Random Linker Substructure:", random_linker)
print("Random E3 Substructure:", random_e3)


Random POI Substructure: [*:1]CNC(=O)c1ccc(N2C(=S)N(c3ccc(C#N)c(C(F)(F)F)c3)C(=O)C2(C)C)cc1F
Random Linker Substructure: [*:2]C(=O)CCCCCCn1cc([*:1])nn1
Random E3 Substructure: [*:2]Oc1cc(-c2scnc2C)ccc1CNC(=O)C1CC(O)CN1C(=O)C(NC(=O)C1(F)CC1)C(C)(C)C


In [16]:
random_protacs_df = generate_random_protac_dataset(unique_poi_substructures, unique_linker_substructures, unique_e3_substructures, num_random_protacs)

In [17]:
#split DataFrame into two DataFrames at row 6
val_fraction = 0.15
num_random_train_protacs = int(num_random_protacs*(1-val_fraction)+0.5)
num_random_val_protacs = num_random_protacs-num_random_train_protacs

random_protacs_train_df = random_protacs_df.iloc[:num_random_train_protacs]
random_protacs_val_df = random_protacs_df.iloc[num_random_train_protacs:]

In [18]:
PROTACsmiles_df_train_random, Substructuressmiles_df_train_random = prepare_data_set(random_protacs_train_df, p_column='protac_smiles', poi_column='poi', linker_column='linker', e3_column='e3')
PROTACsmiles_df_val_random, Substructuressmiles_df_val_random = prepare_data_set(random_protacs_val_df, p_column='protac_smiles', poi_column='poi', linker_column='linker', e3_column='e3')

/projects/cc/kallberg_knkn308/Code/PROTAC-Splitter_withInternalData/notebooks/GNN/all_functions.py:827: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_set['substructures'] = test_set.apply(lambda row: '.'.join([str(row[poi_column]), str(row[linker_column]), str(row[e3_column])]), axis=1)
/projects/cc/kallberg_knkn308/Code/PROTAC-Splitter_withInternalData/notebooks/GNN/all_functions.py:827: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_set['substructures'] = test_set.apply(lambda row: '.'.join([str(

In [19]:
training_dataset = PROTACDataset(data=PROTACsmiles_df_train_random, substructures=Substructuressmiles_df_train_random, name=f'{dataname}_train_rand', graph_descriptor_list=graph_descriptor_list)
validation_dataset = PROTACDataset(data=PROTACsmiles_df_val_random, substructures=Substructuressmiles_df_val_random, name=f'{dataname}_val_rand', graph_descriptor_list=graph_descriptor_list)

Processing...
100%|██████████| 16973/16973 [09:31<00:00, 29.69it/s]
Done!
Processing...
100%|██████████| 2995/2995 [01:31<00:00, 32.73it/s]
Done!


## Load and prepare test sets

In [20]:
current_directory = os.getcwd()
raw_data_path = os.path.join(current_directory, "..", "..", "data","raw")

test_set_1_non_public_poi = pd.read_csv(raw_data_path +'/test_set_1_non_public_poi.csv')
test_set_1_2_non_public_poi_and_linker = pd.read_csv(raw_data_path +'/test_set_1_2_non_public_poi_and_linker.csv')
test_set_2_non_public_e3 = pd.read_csv(raw_data_path +'/test_set_2_non_public_e3.csv')
test_set_2_2_non_public_e3_and_linker = pd.read_csv(raw_data_path +'/test_set_2_2_non_public_e3_and_linker.csv')
test_set_3_non_public_linker = pd.read_csv(raw_data_path +'/test_set_3_non_public_linker.csv')
test_set_4_non_public_protac = pd.read_csv(raw_data_path +'/test_set_4_non_public_protac.csv')
test_set_5_public_protac = pd.read_csv(raw_data_path +'/test_set_5_public_protac.csv')

test_set_names_core = ['test_set_1_non_public_poi_v1', 'test_set_2_non_public_e3_v1', 'test_set_3__non_public_linker_v1', 'test_set_4_non_public_protac', 'test_set_5_public_protac_v1', 'test_set_1_2_non_public_poi_and_linker', 'test_set_2_2_non_public_e3_and_linker']
test_set_names = [s + dataname for s in test_set_names_core]
test_sets = [test_set_1_non_public_poi, test_set_2_non_public_e3, test_set_3_non_public_linker, test_set_4_non_public_protac, test_set_5_public_protac, test_set_1_2_non_public_poi_and_linker, test_set_2_2_non_public_e3_and_linker]


In [21]:
test_set_1_protacs, test_set_1_substructures = prepare_data_set(test_sets[0], 'Structure', 'POI Smiles R', 'linker', 'e3 Smiles R')
test_set_2_protacs, test_set_2_substructures = prepare_data_set(test_sets[1], 'Structure', 'POI Smiles R', 'linker', 'e3 Smiles R')
test_set_3_protacs, test_set_3_substructures = prepare_data_set(test_sets[2], 'Structure', 'POI Smiles R', 'linker', 'e3 Smiles R')
test_set_4_protacs, test_set_4_substructures = prepare_data_set(test_sets[3], 'Structure', 'POI Smiles R', 'linker', 'e3 Smiles R')
test_set_5_protacs, test_set_5_substructures = prepare_data_set(test_sets[4], 'Structure', 'POI Smiles R', 'linker', 'e3 Smiles R')
test_set_1_2_protacs, test_set_1_2_substructures = prepare_data_set(test_sets[5], 'Structure', 'POI Smiles R', 'linker', 'e3 Smiles R')
test_set_2_2_protacs, test_set_2_2_substructures = prepare_data_set(test_sets[6], 'Structure', 'POI Smiles R', 'linker', 'e3 Smiles R')

test_dataset_1 = PROTACDataset(data=test_set_1_protacs, substructures=test_set_1_substructures, name=test_set_names[0])
test_dataset_2 = PROTACDataset(data=test_set_2_protacs, substructures=test_set_2_substructures, name=test_set_names[1])
test_dataset_3 = PROTACDataset(data=test_set_3_protacs, substructures=test_set_3_substructures, name=test_set_names[2])
test_dataset_4 = PROTACDataset(data=test_set_4_protacs, substructures=test_set_4_substructures, name=test_set_names[3])
test_dataset_5 = PROTACDataset(data=test_set_5_protacs, substructures=test_set_5_substructures, name=test_set_names[4])
test_dataset_1_2 = PROTACDataset(data=test_set_1_2_protacs, substructures=test_set_1_2_substructures, name=test_set_names[5])
test_dataset_2_2 = PROTACDataset(data=test_set_2_2_protacs, substructures=test_set_2_2_substructures, name=test_set_names[6])


Processing...
100%|██████████| 2078/2078 [00:50<00:00, 41.06it/s]
Done!
Processing...
100%|██████████| 46/46 [00:00<00:00, 46.40it/s]
Done!
Processing...
100%|██████████| 369/369 [00:08<00:00, 44.33it/s]
Done!
Processing...
100%|██████████| 732/732 [00:16<00:00, 43.51it/s]
Done!
Processing...
100%|██████████| 19/19 [00:00<00:00, 42.21it/s]
Done!
Processing...
100%|██████████| 2263/2263 [00:52<00:00, 42.73it/s]
Done!
Processing...
100%|██████████| 147/147 [00:03<00:00, 45.73it/s]
Done!
